In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import requests


In [ ]:
BASE_URL = "https://www.thebluealliance.com/api/v3"


def get_tba_session() -> requests.Session:
    api_key = ""#ADD YOUR TBA API KEY HERE
    if not api_key:
        raise RuntimeError(
            "Set TBA_API_KEY or THE_BLUE_ALLIANCE_API_KEY before running this notebook."
        )

    session = requests.Session()
    session.headers.update(
        {
            "X-TBA-Auth-Key": api_key,
            "User-Agent": "match-loader/1.0",
            "Accept": "application/json",
        }
    )
    return session


def fetch_json(session: requests.Session, path: str):
    response = session.get(f"{BASE_URL}{path}", timeout=30)
    response.raise_for_status()
    return response.json()


def coerce_json_cells(df: pd.DataFrame) -> pd.DataFrame:
    def encode(value):
        if isinstance(value, (dict, list)):
            return json.dumps(value, separators=(",", ":"), sort_keys=True)
        return value

    for column in df.columns:
        if df[column].map(lambda value: isinstance(value, (dict, list))).any():
            df[column] = df[column].map(encode)
    return df


In [3]:
year = int(input("Enter the FRC season year to export: ").strip())
session = get_tba_session()

events = fetch_json(session, f"/events/{year}/simple")
matches = []

for event in events:
    event_key = event["key"]
    event_matches = fetch_json(session, f"/event/{event_key}/matches")

    for match in event_matches:
        match["event_key"] = event_key
        match["event_name"] = event.get("name")
        match["event_start_date"] = event.get("start_date")
        match["event_end_date"] = event.get("end_date")

    matches.extend(event_matches)

if not matches:
    raise RuntimeError(f"No matches were found for season {year}.")

df = pd.json_normalize(matches, sep=".")

actual_time = pd.to_numeric(df["actual_time"], errors="coerce") if "actual_time" in df.columns else pd.Series(index=df.index, dtype="float64")
predicted_time = pd.to_numeric(df["time"], errors="coerce") if "time" in df.columns else pd.Series(index=df.index, dtype="float64")
df["played_time_unix"] = actual_time.fillna(predicted_time)
df["played_time_utc"] = pd.to_datetime(df["played_time_unix"], unit="s", utc=True, errors="coerce")

sort_columns = [column for column in ["played_time_utc", "event_key", "comp_level", "set_number", "match_number", "key"] if column in df.columns]
df = df.sort_values(sort_columns, na_position="last").reset_index(drop=True)
df.insert(0, "played_index", df.index + 1)
df = coerce_json_cells(df)

output_path = Path.cwd() / f"frc_matches_{year}.csv"
df.to_csv(output_path, index=False)
print(f"Wrote {len(df)} matches to {output_path}")


Wrote 23884 matches to c:\Users\Aarush\Documents\Summer2026proj\frc_matches_2025.csv
